# Stacking

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_squared_error
from sklearn.ensemble import StackingClassifier, StackingRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

In [ ]:
# Carregar dados pré-processados
airline_train = pd.read_csv('dados/airline_train_clean.csv')
airline_test = pd.read_csv('dados/airline_test_clean.csv')
X_train_c, y_train_c = airline_train.drop(columns=['satisfaction']), airline_train['satisfaction']
X_test_c, y_test_c = airline_test.drop(columns=['satisfaction']), airline_test['satisfaction']

taxi_train = pd.read_csv('dados/taxi_train_clean.csv')
taxi_test = pd.read_csv('dados/taxi_test_clean.csv')
X_train_r, y_train_r = taxi_train.drop(columns=['Trip_Price']), taxi_train['Trip_Price']
X_test_r, y_test_r = taxi_test.drop(columns=['Trip_Price']), taxi_test['Trip_Price']

## Classificação - Airline

In [ ]:
modelo_c = StackingClassifier(
    estimators=[
        ('dt', DecisionTreeClassifier(random_state=42)),
        ('knn', KNeighborsClassifier()),
    ],
    final_estimator=LogisticRegression(max_iter=1000),
)
modelo_c.fit(X_train_c, y_train_c)

In [ ]:
y_pred = modelo_c.predict(X_test_c)
print('Acurácia:', accuracy_score(y_test_c, y_pred))
print(classification_report(y_test_c, y_pred))

## Regressão - Taxi

In [ ]:
modelo_r = StackingRegressor(
    estimators=[
        ('dt', DecisionTreeRegressor(random_state=42)),
        ('knn', KNeighborsRegressor()),
    ],
    final_estimator=LinearRegression(),
)
modelo_r.fit(X_train_r, y_train_r)

In [ ]:
y_pred = modelo_r.predict(X_test_r)
print('R²:', r2_score(y_test_r, y_pred))
print('RMSE:', np.sqrt(mean_squared_error(y_test_r, y_pred)))

## Comparação entre as técnicas

| Técnica | Acurácia (Airline) | R² (Taxi) | RMSE (Taxi) |
|---|---|---|---|
| Bagging | 0,949 | 0,942 | 11,60 |
| Boosting | 0,940 | 0,941 | 11,71 |
| Voting | 0,917 | 0,885 | 16,40 |
| Stacking | 0,931 | 0,895 | 15,65 |

Bagging e Boosting, que combinam várias árvores, tiveram o melhor desempenho nos dois problemas. Voting e Stacking ficaram abaixo porque combinam modelos de naturezas diferentes, e alguns deles (regressão linear e KNN) são bem mais fracos que as árvores nesses dados. Mesmo assim, o Stacking superou o Voting, já que aprende como pesar cada modelo em vez de dar o mesmo peso para todos.